In [1]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, bindparam
from datetime import date, datetime, timedelta
from pathlib import Path
import pandas as pd
import os
import urllib3

# Load environment variables from .env file
load_dotenv()

strPresto = ('presto://{username}:{password}@{ipaddress}:{port}/{dbname}/{schema}'
             .format(username=os.getenv('HIVE_SVC_USER'),
                     password=os.getenv('HIVE_SVC_PASS'),
                     ipaddress=os.getenv('HIVE_SVC_ADDRESS'),
                     port=os.getenv('HIVE_SVC_PORT'),
                     dbname=os.getenv('HIVE_SVC_DBNAME'),
                     schema=os.getenv('HIVE_SVC_SCHEMA')))
 
presto_engine = create_engine(strPresto, connect_args={"protocol": "https", "requests_kwargs": {"verify": False}})

# disable certificate warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
# ------------------------------------------------------------------
# Start date and end dates for dateselectors in queries
# ------------------------------------------------------------------

#start_date = '2025-12-13' 
#end_date = '2026-02-06' 

today = date.today()
diff_to_friday = (4 - today.weekday()) % 7  # Mon=0 ... Fri=4

# Go back 7 weeks since we are counting the current week
eight_weeks_ago = today - timedelta(weeks=7) 

# Find the Saturday of that week (Mon=0 ... Sun=6, Sat=5)
days_since_saturday = (eight_weeks_ago.weekday() - 5) % 7

start_saturday = eight_weeks_ago - timedelta(days=days_since_saturday)
end_friday = today + timedelta(days=diff_to_friday)

#format date as string for Presto query
start_date = start_saturday.strftime('%Y-%m-%d')
end_date = end_friday.strftime('%Y-%m-%d')


In [3]:
start_date

'2025-12-27'

In [4]:
# ------------------------------------------------------------------
# ICP Client list as stored in hive.care.expert_performance_metrics.metric (lowercase)
# ------------------------------------------------------------------

icp_client_list = [
    "pss-verizon",
    "pss-at&t",
    "mob-verizon",
    "mob-at&t"
]

In [5]:
# ------------------------------------------------------------------
# Load SQL template for agent query
# ------------------------------------------------------------------

sql_path = "SQL/agent_org_data_week.sql"  # <- make sure this path is correct

with open(sql_path, "r") as f:
    AGENT_SQL_TEMPLATE = f.read()

print("Loaded SQL template:")
print(AGENT_SQL_TEMPLATE[:500], "...")

Loaded SQL template:
SELECT
date_add('day', 4, date_trunc('week', CAST(date AS date))) AS week_ending,
expert_id,
site AS mascot,
icp_client,
tenure_group,
coach,
supervisor_id AS coach_id
from hive.care.expert_performance_metrics
WHERE LOWER(icp_client) IN :icp_client_list 
AND
date between DATE :start_date and DATE :end_date 
--and expert_id IN :expert_ids
GROUP BY 1, 2, 3, 4, 5, 6, 7 ...


In [6]:
### Compile SQL With Literal Binds (Code)

#This uses your proven pattern: bind params + `literal_binds=True`.

#python
# ------------------------------------------------------------------
# Build literal SQL for Presto using SQLAlchemy binds
# This allows us to use expanding=True for metric_list and still
# send flattened literal SQL to Presto.
# ------------------------------------------------------------------

def compile_presto_sql(
    sql_template: str,
    engine,
    start_date,
    end_date,
    icp_client_list,
):
    """
    Creates literal SQL for Presto by binding parameters and compiling
    with literal_binds=True.
    """
    
    stmt = text(sql_template).bindparams(
        bindparam("start_date", value=start_date),
        bindparam("end_date", value=end_date),
        bindparam("icp_client_list", value=list(icp_client_list), expanding=True),
    )

    compiled = stmt.compile(
        engine,
        compile_kwargs={"literal_binds": True}
    )

    return str(compiled)


In [7]:
# ------------------------------------------------------------------
# Query Presto for metrics, client groups, and date range, returning the
# aggregated metrics DataFrame.
# ------------------------------------------------------------------

def query_expert_presto_group(
    start_date,
    end_date,
    icp_client_list,
):
    """
    Execute the Presto query for a list of experts over a date range.

    Returns DataFrame with:
      [expert_id, icp_client, site, mascot, coach, coach_id]
    """

    sql = compile_presto_sql(
        sql_template=AGENT_SQL_TEMPLATE,
        engine=presto_engine,
        start_date=start_date,
        end_date=end_date,
        icp_client_list=icp_client_list,
    )

    # Uncomment to debug generated SQL:
    # print(sql)

    with presto_engine.connect() as conn:
        df = pd.read_sql(sql, conn)

    # Normalize types for downstream joins
    if not df.empty:
        df["icp_client"] = df["icp_client"].astype(str)

    return df


In [8]:
df = query_expert_presto_group(
        start_date,
        end_date,
        icp_client_list,
    )

In [9]:
df

,week_ending,expert_id,mascot,icp_client,tenure_group,coach,coach_id
0,2026-02-20,700528,turtles,PSS-Verizon,61-90,rodnel maravilla,536712
1,2026-02-20,364219,mustangs,PSS-Verizon,180+,kristel flores,304441
2,2026-01-02,690475,bears,PSS-Verizon,121-180,ricardo andres cabana,668950
3,2026-02-13,655323,raptors,PSS-Verizon,180+,levi anthony gamboa,634534
4,2026-02-13,675263,raptors,MOB-AT&T,180+,ryan arsel legaspi,570192
...,...,...,...,...,...,...,...
52883,2025-12-26,582610,mustangs,PSS-Verizon,180+,michael harrison,552704
52884,2026-02-06,667823,lions,PSS-AT&T,180+,april alpanta,537997
52885,2026-01-02,689045,mustangs,PSS-Verizon,91-120,steven garcia,546831
52886,2026-01-02,680879,mustangs,PSS-AT&T,121-180,daliris alicea,600527


In [10]:
# Save to CSV in the same directory

from pathlib import Path

file = Path("../data/raw/weekly/2026-02-16/agents.csv")   # replace with your filename

if file.exists():
    file.unlink()
    df.to_csv("../data/raw/weekly/2026-02-16/agents.csv", index=False)
else:
    df.to_csv("../data/raw/weekly/2026-02-16/agents.csv", index=False)
